# Phase 5: Multi-City AQI Forecasting Pipeline
## Production-Ready Batch Processing for 26 Indian Cities

**Goal:** Scale from single-city (Hyderabad) to all cities in dataset

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from prophet import Prophet
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

print('Imports complete')

In [ ]:
# Cell 2: Get all cities from database
engine = create_engine('postgresql://postgres@localhost:5432/india_air_quality')

cities_df = pd.read_sql("""
    SELECT city, COUNT(*) as total_days, 
           MIN(date) as start_date, 
           MAX(date) as end_date
    FROM city_day 
    WHERE aqi IS NOT NULL
    GROUP BY city 
    ORDER BY total_days DESC
""", engine)

print(f'Found {len(cities_df)} cities')
print(cities_df.head(10))

eligible_cities = cities_df[
    (cities_df['total_days'] >= 1000) & 
    (cities_df['end_date'] >= '2024-01-01')
]['city'].tolist()

print(f'Eligible: {len(eligible_cities)} cities: {eligible_cities}')

In [ ]:
# Cell 3: Reusable forecasting function

def forecast_city(city_name, min_days=1000):
    try:
        query = f"""
            SELECT date as ds, aqi as y
            FROM city_day
            WHERE city = '{city_name}' AND aqi IS NOT NULL
            ORDER BY date
        """
        df = pd.read_sql(query, engine)
        df['ds'] = pd.to_datetime(df['ds'])
        
        if len(df) < min_days:
            return {'city': city_name, 'status': 'insufficient_data', 'days': len(df)}
        
        if df['ds'].max() < '2023-01-01':
            return {'city': city_name, 'status': 'no_recent_data'}
        
        train = df[df['ds'] < '2023-01-01']
        test = df[df['ds'] >= '2023-01-01']
        
        if len(test) < 30:
            return {'city': city_name, 'status': 'insufficient_test'}
        
        model = Prophet(yearly_seasonality=True, weekly_seasonality=False, 
                       daily_seasonality=False, changepoint_prior_scale=0.05)
        model.fit(train)
        
        future = model.make_future_dataframe(periods=len(test), freq='D')
        forecast = model.predict(future)
        
        predictions = forecast[forecast['ds'].isin(test['ds'])][['ds', 'yhat']]
        results = test.merge(predictions, on='ds')
        
        mape = np.mean(np.abs((results['y'] - results['yhat']) / results['y'])) * 100
        rmse = np.sqrt(np.mean((results['y'] - results['yhat'])**2))
        
        # 2030 forecast
        future_2030 = model.make_future_dataframe(periods=365*6, freq='D')
        forecast_2030 = model.predict(future_2030)
        pred_2030 = forecast_2030[forecast_2030['ds'].dt.year == 2030]['yhat'].mean()
        
        recent_avg = df[df['ds'] >= '2024-01-01']['y'].mean()
        trend = 'improving' if pred_2030 < recent_avg else 'worsening'
        
        return {
            'city': city_name,
            'status': 'success',
            'mape': round(mape, 2),
            'rmse': round(rmse, 2),
            'avg_aqi_2024': round(recent_avg, 1),
            'pred_aqi_2030': round(pred_2030, 1),
            'trend': trend
        }
        
    except Exception as e:
        return {'city': city_name, 'status': 'error', 'error': str(e)}

print('Function defined')

In [ ]:
# Cell 4: Process all cities

print(f'Processing {len(eligible_cities)} cities...')

results = []
for city in tqdm(eligible_cities):
    result = forecast_city(city)
    results.append(result)

results_df = pd.DataFrame(results)

success_df = results_df[results_df['status'] == 'success']
print(f'Successful: {len(success_df)}/{len(eligible_cities)}')
print(f'Average MAPE: {success_df["mape"].mean():.1f}%')
print(f'Best: {success_df["mape"].min():.1f}% ({success_df.loc[success_df["mape"].idxmin(), "city"]})')
print(f'Worst: {success_df["mape"].max():.1f}% ({success_df.loc[success_df["mape"].idxmax(), "city"]})')

In [ ]:
# Cell 5: Display results

success_df = success_df.sort_values('mape')

print('CITY RANKINGS:')
print(success_df[['city', 'mape', 'avg_aqi_2024', 'pred_aqi_2030', 'trend']].to_string(index=False))

success_df.to_csv('../outputs/multi_city_results.csv', index=False)
print('\nSaved: outputs/multi_city_results.csv')

In [ ]:
# Cell 6: Visualize

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(success_df['mape'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(success_df['mape'].mean(), color='red', linestyle='--', label=f'Mean: {success_df["mape"].mean():.1f}%')
ax.axvline(15.6, color='green', linestyle='--', label='Hyderabad: 15.6%')
ax.set_xlabel('MAPE (%)')
ax.set_ylabel('Number of Cities')
ax.set_title('Forecast Accuracy Across Cities')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Best and worst
print('Best cities (MAPE < 15%):')
print(success_df[success_df['mape'] < 15][['city', 'mape']].to_string(index=False))

print('\nWorst cities (MAPE > 25%):')
print(success_df[success_df['mape'] > 25][['city', 'mape']].head().to_string(index=False))